# sisl 01 · SIESTA matrices and visualization

[Open in Colab](https://colab.research.google.com/github/YHKlab-RGLee/tutorial/blob/main/sisl/01/Sisl_Matrices.ipynb)

How do the matrices saved by SIESTA become a band structure or an electron density? We follow a four-atom bulk BN calculation from its inputs to these observables, using sisl's public API.

Run the cells in order in a **Colab CPU runtime**. First run SIESTA, then learn the orbital and periodic-cell indexing of H, S, and DM. View their values before forming Bloch matrices, calculating bands and band folding, and projecting states onto real-space grids.

The SZP basis, k mesh, and mesh cutoff keep this exercise small; they are not converged research settings.

## 1. Setup

Use micromamba to install the **SIESTA 5.4.2 serial** executable. Python cells use Colab's Python and **sisl 0.16.4**, installed with pip. No compilation, kernel replacement, or restart is required.

This follows the official [SIESTA conda installation guide](https://docs.siesta-project.org/projects/siesta/en/stable/installation/conda.html). Installation may take a few minutes.

In [ ]:
%%bash
set -e
mkdir -p /content/mamba
curl -Ls --fail https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj -C /content/mamba bin/micromamba
/content/mamba/bin/micromamba create -y -r /content/mamba -p /content/siesta-env     -c conda-forge --override-channels 'siesta=5.4.2=nompi_*'
/content/siesta-env/bin/siesta --version

### Python packages

Use **sisl 0.16.4** in the notebook's Python environment. SIESTA is a separate executable; installing its conda environment does not replace the notebook kernel.

The `viz-plotly` extra installs the interactive Plotly backend and the grid-isosurface dependencies. No external structure viewer is needed.

In [ ]:
%pip install -q "sisl[viz-plotly]==0.16.4" matplotlib netCDF4

### Imports

We use NumPy for arrays and Matplotlib for plots. Limit OpenMP threads before importing sisl to keep this small example suitable for a CPU runtime.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "2"

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import sisl as si
import sisl.viz  # Register .plot() for geometry and grid objects.

print("sisl:", si.__version__)

## 2. Inputs

Keep these four files together: `RUN.fdf` includes `STRUCT.fdf`, and SIESTA looks up the pseudopotentials by species label.

| File | Role in the calculation |
|---|---|
| `RUN.fdf` | Basis, SCF settings, k sampling, and output options |
| `STRUCT.fdf` | Lattice vectors, species, and atomic coordinates |
| `B.psf`, `N.psf` | B and N pseudopotentials |

Download the inputs into a working directory. This cell only prepares files; the next cells view them.

In [ ]:
from urllib.request import urlretrieve

work = Path("/content/bn-matrices")
work.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/YHKlab-RGLee/tutorial/main/sisl/01/src/inputs"
for name in ["RUN.fdf", "STRUCT.fdf", "B.psf", "N.psf"]:
    urlretrieve(f"{base}/{name}", work / name)
os.chdir(work)
print("Working directory:", work)

### Structure: `STRUCT.fdf`

The reference cell contains two B and two N atoms in two layers. Coordinates and lattice vectors are in Å. The in-plane lattice vectors are oblique, so fractional coordinates and Cartesian coordinates are different.

In [ ]:
print(Path("STRUCT.fdf").read_text())

### Atomic structure

Rotate and zoom this 3D view to see the two BN layers and the oblique unit cell. Hover over atoms to identify them. `read_geometry()` reads positions from the FDF inputs; numerical orbitals and an SCF calculation are not needed for a structure plot.

`axes="xyz"` requests three Cartesian axes. Plotting `nsc=[2, 2, 1]` repeats the structure in the plane to reveal the hexagonal network. This is a display choice, separate from matrix-storage `geometry.nsc`; it does not change the calculation.

In [ ]:
structure = si.get_sile("RUN.fdf").read_geometry()
structure_view = structure.plot(axes="xyz", nsc=[2, 2, 1], show_cell="box")
structure_view.show(renderer="colab")

### Settings: `RUN.fdf`

`PAO.BasisSize SZP` chooses a single-zeta basis with polarization orbitals. `MD.NumCGsteps 0` keeps the structure fixed; `DM.UseSaveDM false` starts without reusing a saved density matrix. The final four options request the data that sisl will read.

In [ ]:
print(Path("RUN.fdf").read_text())

## 3. SIESTA calculation

Now run SIESTA on `RUN.fdf`. It iterates the density and Hamiltonian to self-consistency and writes its log to `siesta.out`. Wait for this cell to finish before reading any results. The displayed log lines show convergence and energies; if it fails, view the full log.

In [ ]:
%%bash
set -e
OMP_NUM_THREADS=2 OPENBLAS_NUM_THREADS=1 /content/siesta-env/bin/siesta < RUN.fdf > siesta.out
grep -E 'SCF Convergence|SCF_NOT_CONV|Fermi|Total =' siesta.out | tail -12

### Output files

These are **outputs of SIESTA and inputs to post-processing**. Matrix elements describe relations between basis orbitals; the numerical orbital functions are separately needed to evaluate quantities at spatial positions.

| Output | Requested by | Contents | What it enables |
|---|---|---|---|
| `BN.HSX` | `SaveHS true` | Hamiltonian H, overlap S, and Fermi level | Generalized diagonalization, bands, DOS, eigenstates |
| `BN.DM` | `Write.DM true` | Converged density-matrix coefficients | Reconstructing electron density together with the numerical orbitals |
| `B.ion.xml`, `N.ion.xml` | `WriteIonXML true` | Numerical atomic-orbital basis, including radial functions | Evaluating density and wavefunctions on spatial grids |
| `BN.RHO` | `SaveRho true` | Electron density on SIESTA's grid | Comparing a reconstructed density with the original result |

The check below stops early if SCF did not converge or an expected file is missing. File existence alone does not establish convergence, and SCF convergence alone does not establish convergence of the basis or k mesh.

In [ ]:
required = ["BN.HSX", "BN.DM", "BN.RHO", "B.ion.xml", "N.ion.xml"]
missing = [name for name in required if not Path(name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing SIESTA outputs: {missing}; read siesta.out")
log = Path("siesta.out").read_text()
if "SCF Convergence" not in log or "SCF_NOT_CONV" in log:
    raise RuntimeError("SCF did not converge; read siesta.out before continuing")
print("Converged SCF outputs are ready for post-processing.")

## 4. Matrices

### Hamiltonian H

$H_{ij}(\mathbf R)=\langle\phi_{i0}|\hat H|\phi_{j\mathbf R}\rangle$ couples an orbital in the reference cell to an orbital in a translated cell. Its units are eV. Diagonalization gives energies and eigenvectors.

`get_sile("RUN.fdf")` selects the FDF reader, which connects the structure, basis, and output files. `order=["HSX"]` selects the saved HSX format. The object summary includes geometry and storage information; we will decode its dimensions below.

In [ ]:
fdf = si.get_sile("RUN.fdf")
H = fdf.read_hamiltonian(order=["HSX"])
print(H)

### Overlap S

$S_{ij}(\mathbf R)=\langle\phi_{i0}|\phi_{j\mathbf R}\rangle$ is dimensionless. SIESTA's atomic orbitals are nonorthogonal: different orbitals can overlap, so S is not generally the identity. Energies therefore satisfy $H(k)c=E S(k)c$.

H already carries overlap for diagonalization. Read S separately here to view the basis metric itself.

In [ ]:
S = fdf.read_overlap(order=["HSX"])
print(S)

### Density matrix DM

The density matrix combines the occupied eigenstates and their occupations. Its coefficients weight products of basis orbitals when reconstructing the electron density. It is neither an energy matrix nor a spatial density grid. In a nonorthogonal basis, its diagonal alone is not a list of atomic electron populations.

Read the converged coefficients from `BN.DM`; the FDF reader attaches the geometry and numerical basis.

In [ ]:
DM = fdf.read_density_matrix(order=["DM"])
print(DM)

### Fermi level

Read $E_F$ from the same SCF calculation. We will plot eigenvalues as $E-E_F$; subtracting this scalar changes the energy reference, not the eigenvectors.

In [ ]:
Ef = si.get_sile("BN.HSX").read_fermi_level()
print("Fermi level [eV]:", Ef)

## 5. Matrix dimensions

### Atoms and orbitals

Matrix indices refer to **basis orbitals**, not atoms. For these B/N SZP inputs, each atom has one s orbital, three p orbitals, and five polarization d orbitals: 9 per atom and 36 in the four-atom cell. This count depends on the chosen basis.

Use `a2o(..., all=True)` to map each zero-based atom index to its orbitals. This also identifies the atom groups visible in matrix plots.

In [ ]:
geometry = H.geometry
for ia, atom in enumerate(geometry.atoms):
    print(f"atom {ia} ({atom.symbol}): orbitals {geometry.a2o(ia, all=True)}")
print(f"Reference cell: {geometry.na} atoms, {geometry.no} orbitals")

### Cell offsets

A periodic operator has infinitely many translated copies. Localized orbitals allow sisl to store a finite set of neighboring-cell blocks:

$$M_{i,j+N_o b}=M_{ij}(\mathbf R_b),\qquad
M=[M(0)\;M(\mathbf R_1)\;\cdots].$$

Rows select the $N_o$ reference-cell orbitals. Column `j + no * b` selects orbital `j` in cell-offset slot `b`. Each block is $N_o\times N_o$.

`nsc` counts stored cell-offset slots along each lattice direction; it is **not the k mesh or the number of physical reference cells**. Here `[7, 7, 3]` gives $N_s=147$ slots, so one matrix component is $36\times(36\cdot147)=36\times5292$. Some blocks are zero. The actual offset ordering is given by `lattice.sc_off`; do not guess a block number from its direction.

In [ ]:
no = geometry.no
n_cells = geometry.lattice.n_s
print("nsc:", geometry.nsc, "→ cell-offset slots:", n_cells)
print("One component:", (no, no * n_cells))
print("First cell offsets:\n", geometry.lattice.sc_off[:5])

### Components

The objects report three axes: `(reference orbitals, orbitals across cell offsets, components)`. For this **non-spin-polarized, nonorthogonal** calculation:

| Object | Shape | Component meaning |
|---|---|---|
| H | `(36, 5292, 2)` | Hamiltonian values, plus overlap |
| S | `(36, 5292, 1)` | Overlap values only |
| DM | `(36, 5292, 2)` | Density-matrix values, plus an overlap storage slot |

The extra component in H or DM is not an extra orbital or a second spin channel. A DM file supplies density coefficients; an overlap storage slot does not by itself guarantee populated overlap data. Use the HSX-derived S when you need overlap.

`tocsr(dim=0)` extracts the physical values into a two-dimensional sparse matrix. These dimensions describe this input, not all SIESTA calculations; changing the basis or orbital ranges can change them.

In [ ]:
for name, matrix in [("H", H), ("S", S), ("DM", DM)]:
    print(f"{name}: object {matrix.shape}, component 0 {matrix.tocsr(dim=0).shape}")

## 6. Real-space matrices

### Full matrices

Plot the actual signed values of H, S, and DM, including zeros. `imshow` with a diverging scale shows magnitude and sign; a sparsity plot would show only where entries occur. Each panel has its own symmetric color scale because the quantities have different meanings and units: H is in eV, S is dimensionless, and DM contains dimensionless occupation coefficients.

The arrays are very wide: the horizontal axis concatenates 147 blocks, while the vertical axis has only 36 orbitals. `aspect="auto"` compresses columns to show the whole layout. Use the block views below to resolve individual elements. Dense conversion is convenient for this small example, but can be expensive for large systems.

In [ ]:
matrices = [H, S, DM]
labels = ["H [eV]", "S", "DM"]
fig, axes = plt.subplots(3, 1, figsize=(12, 7), constrained_layout=True)
for ax, matrix, label in zip(axes, matrices, labels):
    values = matrix.tocsr(dim=0).toarray()
    limit = np.abs(values).max()
    im = ax.imshow(values, origin="lower", aspect="auto", interpolation="nearest",
                   cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set(title=label, xlabel="j + no × cell-offset slot", ylabel="reference orbital i")
    fig.colorbar(im, ax=ax)

### Cell blocks

`isc = [0, 0, 0]` selects couplings within the reference cell; `[1, 0, 0]` selects couplings to the cell translated by the first lattice vector. `sc_index(isc)` converts that integer translation to the correct stored block index. Try another offset within the stored range.

Below, each column compares these two offsets with the **same color scale for that quantity**. Lines separate atoms (9 orbitals each here). Within-cell diagonal entries describe same-orbital matrix elements; off-diagonal entries describe orbital couplings, overlaps, or density coefficients. A diagonal H element alone is not a band energy: bands depend on the full Hamiltonian and overlap.

An individual nonzero-offset block need not be symmetric: its transpose belongs to the opposite offset, $M(\mathbf R)=M(-\mathbf R)^\dagger$.

In [ ]:
offsets = [[0, 0, 0], [1, 0, 0]]
fig, axes = plt.subplots(2, 3, figsize=(12, 7), constrained_layout=True)
for col, (matrix, label) in enumerate(zip(matrices, labels)):
    blocks = []
    for isc in offsets:
        block_index = matrix.geometry.sc_index(isc)
        columns = slice(block_index * no, (block_index + 1) * no)
        blocks.append(matrix.tocsr(dim=0)[:, columns].toarray())
    limit = max(np.abs(block).max() for block in blocks)
    for row, (isc, block) in enumerate(zip(offsets, blocks)):
        ax = axes[row, col]
        im = ax.imshow(block, origin="lower", interpolation="nearest",
                       cmap="RdBu_r", vmin=-limit, vmax=limit)
        for boundary in geometry.firsto[1:-1] - 0.5:
            ax.axhline(boundary, color="gray", lw=0.4)
            ax.axvline(boundary, color="gray", lw=0.4)
        ax.set(title=f"{label}, isc={isc}", xlabel="orbital j in selected cell",
               ylabel="reference orbital i")
    fig.colorbar(im, ax=axes[:, col], shrink=0.8)

## 7. Bloch matrices

At a chosen wavevector, Bloch phases combine the neighbor-cell blocks:

$$
H(\mathbf k)=\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}H(\mathbf R),
\qquad
S(\mathbf k)=\sum_{\mathbf R}e^{i\mathbf k\cdot\mathbf R}S(\mathbf R).
$$

These formulas use sisl's default lattice-vector gauge. If the k coordinates supplied to sisl are fractional reciprocal coordinates $\mathbf q$ and the integer cell offset is $\mathbf n$, then $\mathbf k\cdot\mathbf R=2\pi\mathbf q\cdot\mathbf n$.

$$
\underbrace{[\ H(\mathbf0)\;H(\mathbf R_1)\;\cdots\ ]}_{36\times5292}
\quad\xrightarrow{\text{Bloch sum at one }\mathbf k}\quad
\underbrace{H(\mathbf k)}_{36\times36}.
$$

**At Γ, every phase is 1, but all neighbor-cell blocks still contribute:**

$$H(\Gamma)=\sum_{\mathbf R}H(\mathbf R)\ne H(\mathbf0)\quad\text{in general}.$$

Thus a Γ-point calculation still includes periodic connections. It does not isolate the reference cell. The square matrix describes the same 36 basis orbitals at one Bloch wavevector; no orbitals have been removed. `H.Hk()` and `H.Sk()` perform these sums directly.

### Γ-point matrices

At Γ, H(k) and S(k) are real (up to numerical precision) and square. Display their signed values on orbital axes. These are sums over all stored offsets, not just the `[0, 0, 0]` blocks plotted above.

In [ ]:
k = [0.0, 0.0, 0.0]
Hk = H.Hk(k=k, format="array")
Sk = H.Sk(k=k, format="array")
fig, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for ax, matrix, title in zip(axes, [Hk, Sk], ["H(Γ) [eV]", "S(Γ)"]):
    limit = np.abs(matrix.real).max()
    im = ax.imshow(matrix.real, origin="lower", cmap="RdBu_r",
                   vmin=-limit, vmax=limit, interpolation="nearest")
    ax.set(title=title, xlabel="orbital j", ylabel="orbital i")
    fig.colorbar(im, ax=ax)

### Eigenvalues

There are 36 eigenvalues at each k, one per reference-cell orbital in this nonmagnetic calculation. `H.eigh()` uses the stored overlap to solve $H(k)c=E S(k)c$. Calling `np.linalg.eigh(Hk)` alone would incorrectly assume an orthonormal basis.

The eigenvalues below are measured relative to the SCF Fermi level.

In [ ]:
eigenvalues = H.eigh(k=k)
print("Number of Γ eigenvalues:", len(eigenvalues))
print("Γ eigenvalues − EF [eV]:\n", eigenvalues - Ef)

## 8. Band structure

Sample the same eigenvalue problem along Γ–M–K–Γ. `BandStructure` takes fractional reciprocal-lattice coordinates; `apply.array.eigh()` evaluates H at each point. The resulting array has one row per k point and one column per band. No new SCF calculation or `.bands` file is needed.

In [ ]:
points = np.array([[0, 0, 0], [0.5, 0, 0], [1/3, 1/3, 0], [0, 0, 0]])
names = ["Γ", "M", "K", "Γ"]
bs = si.BandStructure(H, points, 121, names=names)
bands = bs.apply.array.eigh() - Ef
x = bs.lineark()
ticks, labels = bs.lineartick()
print("k points × bands:", bands.shape)

### Band plot

Each curve follows one sorted eigenvalue along the path. Zero is the saved Fermi level. The plot shows a selected energy window; it does not display all 36 bands.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, bands, color="tab:blue", lw=0.8)
plt.xticks(ticks, labels)
plt.axhline(0, color="gray", ls=":")
plt.ylabel("E − EF [eV]")
plt.ylim(-15, 15)
plt.xlim(x[0], x[-1])
plt.grid(axis="x")

## 9. Supercells

`H.tile(2, axis=0)` doubles the first lattice vector and repeats atoms, orbitals, Hamiltonian connections, and overlap. The reference cell now has 72 orbitals, so it has 72 bands per k point. Existing couplings are remapped, including connections across the new cell boundaries. This preserves the crystal without a new SCF calculation.

Unlike `nsc`, which describes neighboring-cell storage, `tile()` changes the reference cell itself. It does not produce a self-consistent Hamiltonian for a defect or a strained crystal.

In [ ]:
H2 = H.tile(2, axis=0)
print("Reference-cell orbitals:", H.no, "→", H2.no)
print("Stored shapes:", H.shape, "→", H2.shape)

### k-point mapping

Doubling $a_1$ halves $b_1$. Original fractional coordinates $(q_1,q_2,q_3)$ become $(2q_1,q_2,q_3)$ in the supercell at the same Cartesian k. Using identical fractional coordinates would compare different physical paths.

We keep the original Γ–M–K–Γ path in an extended-zone representation. These labels refer to the original cell, not the supercell's M and K points.

In [ ]:
bs2 = si.BandStructure(H2, points * [2, 1, 1], 121, names=names)
bands2 = bs2.apply.array.eigh() - Ef
print("k points × bands:", bands.shape, "→", bands2.shape)

### Supercell bands

The original bands (black dashed) lie among the supercell bands (orange) along the same physical path. The extra branches arise from the smaller reciprocal period; they are not new electronic states of a different material.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(x, bands2, color="tab:orange", lw=1, alpha=0.7)
plt.plot(x, bands, color="black", lw=0.6, ls="--")
plt.xticks(ticks, labels)
plt.ylabel("E − EF [eV]")
plt.title("2× cell (orange), original cell (black): same physical k")
plt.ylim(-15, 15)
plt.xlim(x[0], x[-1])
plt.grid(axis="x")

### Band folding

The path $q_1=-1/4\ldots1/4$, with the other coordinates zero, spans one reciprocal period of the supercell along this direction. Combining the original-cell eigenvalues $E(q)$ and $E(q+(1/2,0,0))$ gives all supercell eigenvalues. Sorting the combined eigenvalues lets us check this statement numerically.

In [ ]:
fold = si.BandStructure(H, [[-0.25, 0, 0], [0.25, 0, 0]], 81)
fold2 = si.BandStructure(H2, [[-0.5, 0, 0], [0.5, 0, 0]], 81)
e0 = fold.apply.array.eigh()
e1 = np.array([H.eigh(k=q + [0.5, 0, 0]) for q in fold.k])
e2 = fold2.apply.array.eigh()
expected = np.sort(np.concatenate([e0, e1], axis=1), axis=1)
print("folding error [eV]:", np.max(np.abs(e2 - expected)))
np.testing.assert_allclose(e2, expected, atol=1e-5)

### Folded bands

Orange supercell eigenvalues should follow the union of the original q branch (black dashed) and the shifted q+½ branch (blue dotted). Doubling the reference cell changes how states are labelled, while preserving their energies.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(fold.k[:, 0], e2 - Ef, color="tab:orange", lw=2)
plt.plot(fold.k[:, 0], e0 - Ef, "k--", lw=0.7)
plt.plot(fold.k[:, 0], e1 - Ef, "b:", lw=0.7)
plt.xlabel("q₁ in original reciprocal coordinates")
plt.ylabel("E − EF [eV]")
plt.title("Supercell = original q branch + shifted q+½ branch")
plt.ylim(-15, 15)
plt.xlim(-0.25, 0.25)

## 10. Electron density

DM has orbital indices; a `Grid` has spatial sampling indices. To connect them, we need both the saved density coefficients and numerical orbitals from `B.ion.xml` and `N.ion.xml`. Pseudopotential files alone cannot provide those functions.

In a real orbital basis, the periodic density can be written as

$$n(\mathbf r)=\sum_{ij}\sum_{\mathbf T,\mathbf R}
D_{ij}(\mathbf R)\phi_i(\mathbf r-\mathbf T)
\phi_j(\mathbf r-\mathbf T-\mathbf R).$$

Here T runs over translated reference cells and R is the relative cell offset stored in DM. Only orbitals whose finite support reaches r contribute. `DM.density()` evaluates these products on the grid.

The nonmagnetic DM already includes spin occupations: do not multiply by two. Two B atoms (3 valence electrons each) and two N atoms (5 each) give **16 electrons per reference cell**. Neighbor-cell storage does not multiply this count.

### Density grid

`Grid(0.25, ...)` requests a target spacing of 0.25 Å; the actual spacing is adjusted to fit an integer number of points in the cell. `DM.density()` adds to the grid, so create a fresh grid on each evaluation.

The result is electron number density in e/Å³. Multiplying its sum by the volume per point approximates the electron count. The cube file can be opened in a volumetric viewer.

In [ ]:
rho = si.Grid(0.25, geometry=DM.geometry)
DM.density(rho)
print("grid shape:", rho.shape)
print("integrated electrons:", rho.grid.sum() * rho.dvolume)
rho.write("BN-density.cube")

### Grid convergence

Here we evaluate the same DM and PAOs on different real-space grids; the SCF `MeshCutoff` stays fixed. Check whether the integrated density approaches 16 electrons as the grid becomes finer. Convergence need not be monotonic.

In [ ]:
grids = []
for spacing in [0.40, 0.25, 0.15]:
    grid = si.Grid(spacing, geometry=DM.geometry)
    DM.density(grid)
    grids.append(grid)
    electrons = grid.grid.sum() * grid.dvolume
    print(f"{spacing:.2f} Å : {grid.shape}, N = {electrons:.6f}, ΔN = {electrons - 16:+.6f}")

### 3D density

A 2D slice shows one plane; a 3D **isosurface** shows where the density equals a chosen value throughout the cell. This connects the grid to the atomic positions and layer geometry.

Use the finest grid from the convergence comparison. `isos` sets the surface level in e/Å³ and its appearance; `plot_geom=True` overlays atoms. The surface is a threshold of the electron density, not an atomic boundary. Lower `density_level` to include lower-density regions between atoms. The underlying grid and integrated electron count stay unchanged.

In [ ]:
density_level = 0.2  # electrons / Å³
rho_view = grids[-1].plot(
    axes="xyz", plot_geom=True,
    isos=[dict(val=density_level, color="orange", opacity=0.5,
               name=f"n = {density_level} e/Å³")],
)
rho_view.show(renderer="colab")

### Density slices

The in-plane lattice vectors are oblique. The plot axes show **fractional coordinates $f_1,f_2$**, rather than Cartesian x/y. For each grid, plot the slice nearest the first BN layer ($f_3=1/4$), using the same color scale.

In [ ]:
vmax = max(g.grid.max() for g in grids)
fig, axes = plt.subplots(1, 3, figsize=(12, 3), constrained_layout=True)
for ax, grid in zip(axes, grids):
    iz = round(0.25 * grid.shape[2])
    im = ax.imshow(grid.grid[:, :, iz].T, origin="lower", extent=[0, 1, 0, 1],
                   vmin=0, vmax=vmax, aspect="auto")
    ax.set(title=f"{grid.shape}, f₃={iz / grid.shape[2]:.3f}", xlabel="f₁", ylabel="f₂")
fig.colorbar(im, ax=axes, label="electrons / Å³")

### SIESTA density comparison

This comparison additionally needs `BN.RHO`, saved by `SaveRho true`. Use the original four-atom `DM` and its numerical orbitals.

The sisl reader converts SIESTA's stored units. `Grid(reference.shape, lattice=reference.lattice)` creates a fresh array with the same shape, cell, and origin, allowing a pointwise comparison. View both the integrated electron counts and the pointwise RMS difference. PAO interpolation and numerical integration can introduce small differences.

In [ ]:
reference = si.get_sile("BN.RHO").read_grid()
reconstructed = si.Grid(reference.shape, lattice=reference.lattice)
DM.density(reconstructed)
difference = reconstructed.grid - reference.grid
print("SIESTA N:", reference.grid.sum() * reference.dvolume)
print("sisl N:", reconstructed.grid.sum() * reconstructed.dvolume)
print("RMS [e/Å³]:", np.sqrt(np.mean(difference**2)))
print("max |difference| [e/Å³]:", np.max(np.abs(difference)))

## 11. Density of states

A band path samples selected directions; a DOS samples the Brillouin zone with weights. Reuse H and the same energy reference, and average the broadened eigenvalues over a `MonkhorstPack` mesh. `wrap` turns each eigenstate result into a DOS before averaging.

This nonmagnetic Hamiltonian gives a DOS **per spin channel**; multiply by two for both spins. The mesh and broadening affect smoothness and convergence. The plotted energy window contains only part of the full spectrum.

In [ ]:
bz = si.MonkhorstPack(H, [6, 6, 2])
energy = np.linspace(-15, 15, 301)
dos = bz.apply.average.eigenstate(wrap=lambda state: state.DOS(energy + Ef))
plt.figure(figsize=(6, 3))
plt.plot(energy, dos)
plt.xlabel("E − EF [eV]")
plt.ylabel("DOS [states/eV/cell/spin]")

## 12. Wavefunctions

### k point and band

A wavefunction belongs to a **particular k point and band**. Here we choose Γ, `k_wave = [0, 0, 0]`, in fractional reciprocal coordinates, and `band_index = 7`, the **eighth band** in ascending energy order (Python indices start at zero).

`H.eigenstate(k=k_wave)` solves all 36 bands at that k point. `.sub(band_index)` then keeps one eigenvalue and its orbital coefficients; it does not choose an atom or an orbital. The selected state retains its k-point information for spatial projection.

For this nonmagnetic insulator, 16 electrons occupy eight spin-degenerate bands, so index 7 is the highest occupied band at Γ. Index 8 selects the first unoccupied band there. This does not locate the valence-band maximum across the whole Brillouin zone.

Change `k_wave` and `band_index` below, then rerun the following wavefunction cells. For example, `[1/3, 1/3, 0]` selects K on the band path. The grid uses complex values so it also works away from Γ.

Version note: sisl 0.16.4 emits `k != Gamma is currently untested!` for non-Γ projection. Γ is the main example here; treat other k points as an extension and check normalization and convergence.

In [ ]:
k_wave = [0.0, 0.0, 0.0]  # Γ; fractional reciprocal coordinates
band_index = 7             # Eighth band; zero-based index
states = H.eigenstate(k=k_wave)
state = states.sub(band_index)
print(f"k = {k_wave}, band index = {band_index} (band {band_index + 1})")
print("E − EF [eV]:", state.eig[0] - Ef)

### Wavefunction grid

`state.wavefunction(psi)` combines the selected coefficients with the numerical orbitals and the state's Bloch phases on a fresh grid. The `B.ion.xml` / `N.ion.xml` basis attached to H is needed here.

Use a complex grid for arbitrary k. The probability density is $|\psi|^2$, not the real part of $\psi$ squared. Its cell integral should approach 1 as the grid is refined; this single-state density is different from the total occupied-state density with 16 electrons. The cube export below contains only the real part of the wavefunction.

In [ ]:
psi = si.Grid(0.15, geometry=H.geometry, dtype=complex)
state.wavefunction(psi)
probability = np.abs(psi.grid)**2
print("∫ |ψ|² dV:", probability.sum() * psi.dvolume)
psi.write("BN-state-real.cube", imag=False)

### Probability slice

This slice shows the selected state's $|\psi|^2$ near fractional height $f_3=1/4$. The in-plane axes are fractional coordinates, as in the total-density slices. A single state's spatial pattern can differ strongly from the total electron density.

In [ ]:
iz = round(0.25 * psi.shape[2])
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
im = ax.imshow(probability[:, :, iz].T, origin="lower", extent=[0, 1, 0, 1],
               aspect="auto")
ax.set(xlabel="f₁", ylabel="f₂", title=f"k={k_wave}, band {band_index + 1}: |ψ|²")
fig.colorbar(im, ax=ax, label="probability / Å³")

### 3D wavefunction

At Γ, this real Hamiltonian permits real eigenvectors. The two colored isosurfaces show positive and negative lobes of the selected wavefunction, making relative signs and nodes visible. The level below is 20% of the largest absolute real amplitude; it controls only the displayed surface.

Away from Γ the wavefunction is generally complex: `represent="real"` shows only its real part in the chosen phase convention. An overall sign or phase is arbitrary, so the colors are not positive and negative electrical charge. Within degenerate bands, an individual state's shape can also depend on the eigenvector basis returned by diagonalization.

In [ ]:
amplitude_level = 0.2 * np.abs(psi.grid.real).max()
wave_view = psi.plot(
    axes="xyz", represent="real", plot_geom=True,
    isos=[dict(val=amplitude_level, color="red", opacity=0.6, name="Re ψ > 0"),
          dict(val=-amplitude_level, color="blue", opacity=0.6, name="Re ψ < 0")],
)
wave_view.show(renderer="colab")

### 3D probability density

The probability density $|\psi|^2$ is invariant under an overall phase change and can be visualized for any k point. Copy its values into a real `Grid` with the same cell and geometry, then plot a positive isosurface. It shows where this state has appreciable weight, but loses the sign and phase information of the wavefunction.

A level of 20% of the maximum is a display threshold; it does **not** enclose 20% of the probability. Compare the spatial distribution with the total electron-density surface.

In [ ]:
probability_grid = si.Grid(psi.shape, geometry=H.geometry)
probability_grid.grid[:] = probability
probability_level = 0.2 * probability.max()
probability_view = probability_grid.plot(
    axes="xyz", plot_geom=True,
    isos=[dict(val=probability_level, color="purple", opacity=0.5,
               name=f"|ψ|² = {probability_level:.3f} Å⁻³")],
)
probability_view.show(renderer="colab")

## Exercises

- Change to `H.tile(3, axis=0)`. How do the reciprocal-coordinate transformation and the number of folded branches change?
- Form `H.Hk(k=q)` and `H.Sk(k=q)` away from Γ. Why are they complex but still Hermitian?
- Select `isc=[-1, 0, 0]`. Compare its block with the transpose of the `[1, 0, 0]` block.
- How do the electron-count error and runtime change as you refine the density grid?
- Use `state.PDOS(energy + Ef)` to obtain orbital-resolved DOS. Comparing with the total DOS requires all eigenstates and a k-point average, rather than a single state.

- Change `k_wave` to K and compare band indices 7 and 8. How do their probability distributions differ?
- Vary the density isovalue. Which apparent connections change even though the density grid is unchanged?

## References

- [sisl SIESTA I/O: FDF reader](https://zerothi.github.io/sisl/api/io/generated/sisl.io.siesta.fdfSileSiesta.html)
- [Hamiltonian: Hk, Sk, eigh, tile](https://zerothi.github.io/sisl/api/generated/sisl.physics.Hamiltonian.html)
- [BandStructure](https://zerothi.github.io/sisl/api/generated/sisl.physics.BandStructure.html)
- [DensityMatrix: density](https://zerothi.github.io/sisl/api/generated/sisl.physics.DensityMatrix.html)
- [Official sisl SIESTA tutorial](https://zerothi.github.io/sisl/tutorials/tutorial_siesta_1.html)
- [SIESTA input/output options](https://docs.siesta-project.org/projects/siesta/en/stable/reference/siesta.html)
- [Geometry: orbital indices and cell-offset lookup](https://zerothi.github.io/sisl/api/generated/sisl.Geometry.html)
- [sisl visualization](https://zerothi.github.io/sisl/reference/viz.html)
- [Wavefunction plotting](https://zerothi.github.io/sisl/api/viz/generated/sisl.viz.WavefunctionPlot.html)